In [ ]:
# Загружаем библиотеки

import pandas as pd
import numpy as np
import os

In [ ]:
# Загружаем файлы с данными и соединяем их, создавая единый датасет

folder_path = 'data/data_by_week'
all_data = []

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    data = pd.read_csv(file_path, sep=',', skiprows=3)
    date = filename.split('.')
    date_to_table = '.'.join([date[0], date[1], date[2]])
    data['Date'] = date_to_table
    data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)
    all_data.append(data)
    
data = pd.concat(all_data, ignore_index=True)

In [ ]:
# Удаляем лишние данные

data = data.drop(['Area', 'Department', 'Store_Name', 'Class', 'RSM', 'DC_REC_Season','GrossLCSLSRL3D', 'GrossLCSLSRL7D', 'TurnoverL3D', 'TurnoverL7D','IsStoreVisionStatus'], axis=1)

In [ ]:
# Преобразовываем признак StoreNumber

data['StoreNumber'] = (data['StoreNumber'].astype('str').apply(lambda x: x[4:7])).astype('int')

In [ ]:
# Изменяем название признаков

data = data.rename(columns={
    'StoreNumber' : 'Store_id',
    'Sylecolour' : 'Product_id',
    'SubClass' : 'Category',
    'Gross_LC_ActualPrice' : 'Actual_Price'
})

In [ ]:
# Загружаем данные с информацией о распределении товара

allocation = pd.read_csv('data/allocation_dates.csv', sep=';')


In [ ]:
# Объеденяем датафреймы

data = data.merge(allocation[['First_Allocation_Date', 'Store_id', 'Product_id']], on=['Store_id', 'Product_id'], how='left')

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1492702 entries, 0 to 1492701
Data columns (total 18 columns):
 #   Column                   Non-Null Count    Dtype         
---  ------                   --------------    -----         
 0   Store_id                 1492702 non-null  int32         
 1   Category                 1492702 non-null  object        
 2   Product_id               1492702 non-null  object        
 3   Current_Season           1492702 non-null  object        
 4   Actual_Price             1492702 non-null  object        
 5   Price_Status             1492702 non-null  object        
 6   StockAvailableU          1492702 non-null  int64         
 7   StockUSalesFloor         1492702 non-null  int64         
 8   StockUBackroom           1492702 non-null  int64         
 9   StockinTransitU          1492702 non-null  int64         
 10  StockInProgressU         1492702 non-null  int64         
 11  SLSUL3D                  1492702 non-null  int64         
 12  

In [ ]:
# Производим преобработку временных признаков

data = data[~data['First_Allocation_Date'].isna()]
data['First_Allocation_Date'] = pd.to_datetime(data['First_Allocation_Date'].astype('int').astype('str'), format='%Y%m%d')
data['Days_in_Store'] = (data['Date'] - data['First_Allocation_Date']).dt.days
data = data[(data['Days_in_Store'] >= 0) & (data['Days_in_Store'] <= 20)]

In [ ]:
# Высчитываем продажи за последние 14 дней

def calculate_slsl14d(df):
    df_prev = df[['Store_id', 'Product_id', 'SLSUL7D', 'Date']].copy()
    df_prev.rename(columns={
        'SLSUL7D' : 'SLSUL7D_prev',
        'Date' : 'Date_plus_7d'
    }, inplace=True)
    
    df_prev['Date_plus_7d'] = df_prev['Date_plus_7d'] + pd.Timedelta(days=7)
    
    df_merged = df.merge(
        df_prev, 
        left_on=['Store_id', 'Product_id', 'Date'],
        right_on=['Store_id', 'Product_id', 'Date_plus_7d'],
        how='left'
    )
    
    df_merged['SLSUL14D'] = df_merged['SLSUL7D'] + df_merged['SLSUL7D_prev'].fillna(0)
    df_result = df_merged.drop(columns=['Date_plus_7d'])
    
    return df_result

# Добавляем вычисление в датафрейм

data = calculate_slsl14d(data)

In [ ]:
# Определяем функцию для подсчета скорости продаж

def calculate_speedofsales(df):
    
    # Предварительная проверка на деление на ноль или NaN
    df['Speed_of_Sales'] = df.apply(
        lambda row: round(row['SLSUL14D'] / row['Days_in_Store'], 2)
        if row['Days_in_Store'] < 14 and row['Days_in_Store'] > 0
        else round(row['SLSUL14D'] / 14, 2)
        if row['Days_in_Store'] >= 14
        else 0,
        axis=1
    )
    
    return df

# Производим вычисления

data = calculate_speedofsales(data)

In [ ]:
# Создаем целевую переменную

data['Is_Bestseller'] = data['Speed_of_Sales'].apply(lambda x: 1 if x >= 0.25 else 0)
data['Is_Bestseller'].value_counts(normalize=True)

Is_Bestseller
0    0.820002
1    0.179998
Name: proportion, dtype: float64

In [ ]:
# Удаляем данные о продажи, чтобы избавится от возможной утечки данных

columns_to_drop = ['SLSUL3D', 'SLSUL7D', 'SLSUL7D_prev', 'SLSUL14D', 'Speed_of_Sales']
data = data.drop(columns_to_drop, axis=1)

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153396 entries, 0 to 153395
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   Store_id                 153396 non-null  int32         
 1   Category                 153396 non-null  object        
 2   Product_id               153396 non-null  object        
 3   Current_Season           153396 non-null  object        
 4   Actual_Price             153396 non-null  object        
 5   Price_Status             153396 non-null  object        
 6   StockAvailableU          153396 non-null  int64         
 7   StockUSalesFloor         153396 non-null  int64         
 8   StockUBackroom           153396 non-null  int64         
 9   StockinTransitU          153396 non-null  int64         
 10  StockInProgressU         153396 non-null  int64         
 11  StatusSale               153396 non-null  object        
 12  StatusProductSma

In [ ]:
# Сохраняем датасет

data.to_csv('data/train.csv', index=False)